# 消息类型

In [4]:
from dotenv import load_dotenv
# 用社区依赖引入
from langchain_community.llms import Tongyi

load_dotenv()

model = Tongyi(model="qwen3.6-plus")

In [5]:
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import create_agent

# 创建Agent
agent = create_agent(model="qwen3.6-plus")

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        HumanMessage(content="你好，我是虎哥"),
        AIMessage(content="你好，虎哥，很高兴认识你。"),
        HumanMessage(content="我的名字是什么？")
    ]
})

print(response)

ValueError: Unable to infer model provider for model='qwen3.6-plus'. Please specify 'model_provider' directly.

Supported providers: anthropic, anthropic_bedrock, azure_ai, azure_openai, baseten, bedrock, bedrock_converse, cohere, deepseek, fireworks, google_anthropic_vertex, google_genai, google_vertexai, groq, huggingface, ibm, litellm, mistralai, nvidia, ollama, openai, openrouter, perplexity, together, upstage, xai

For help with specific providers, see: https://docs.langchain.com/oss/python/integrations/providers

我们可以通过遍历Messages数组，更友好的打印结果 for message in response['messages']:
    message.pretty_print()

# 3 多模态消息

In [ ]:
{
    "role": "user",
    "content": [
        {"type": "image", "url": "https://xxx.com/a.jpeg"},
        {"type": "text", "text": "这些图描绘了什么内容？"}
    ]
}

In [16]:
from langchain.chat_models import init_chat_model
import os

# 1.初始化模型
model = init_chat_model(
    model="qwen3.6-plus",  # 这里选择qwen3.6-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 2.创建智能体
agent = create_agent(model=model)

# 3.组织多模态消息
multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
        {"type": "text", "text": "这些图描绘了什么内容？"}
    ])

# 4.调用Agent，发送多模态消息
for token, metadata in agent.stream({
    "messages": [multimodal_message]
}, stream_mode="messages"):
    if token.content:
        print(token.content, end="", flush=True)

这张图片描绘了一个非常温馨、宁静的场景，主要展现了人与宠物在海边的亲密互动。具体内容如下：

1.  **主要人物与动物**：
    *   **一位年轻女性**：她坐在沙滩上，留着长发，身穿一件黑白格纹的衬衫和深色裤子。她面带微笑，神情愉悦地看着对面的狗。
    *   **一只狗**：看起来像是一只拉布拉多犬（或者金毛寻回犬），毛色呈浅金色。它正端坐着，身上戴着带有彩色图案的胸背带。

2.  **动作与互动**：
    *   这是画面的核心：狗狗抬起它的左前爪，女子伸出手握住它的爪子，两人正在玩“握手”的游戏。这个动作体现了人与宠物之间的信任、默契和深厚的感情。
    *   红色的牵引绳被随意地丢在旁边的沙地上，说明他们正在享受放松的时刻。

3.  **环境与氛围**：
    *   **地点**：宽阔的海滩，背景可以看到平静的海面和海浪。
    *   **光线**：光线非常柔和且温暖，呈现出金黄色，从右侧照射过来（看起来像是日落时分或日出时分）。这种逆光效果给画面增添了一种梦幻、温暖的感觉。

总的来说，这张照片捕捉了一个充满爱意和放松的瞬间，展示了人与动物在大自然中和谐共处的美好时光。

本地图片

In [ ]:
{
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {
            "type": "image",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "image/jpeg",
        },
    ]
}

In [ ]:
import base64

# 例如，有一个用户上传的文件，是字节格式img_bytes，我们先将其进行base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])

# 调用Agent，发送消息
response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

4 提示词